# Question Generation for TactfulToM Dataset

This notebook generates various types of questions for the TactfulToM dataset using the functions from `question_generation_utils.py`.

In [1]:
import json
import random
import os
import pandas as pd
from openai import OpenAI
import json
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
!pip install aift

In [2]:
# Import question generation functions from the separate module
from question_generation_utils import (generate_comprehensionQA,generate_justificationQA, 
                                       generate_fact_QA,generate_1stbeliefQAs,
                                       generate_2ndbeliefQAs,generate_infoAccessibilityQAs,
                                       generate_answerabilityQAs,generate_liedetectabilityQAs,
                                       generate_lieabilityQAs,load_justification_options,merge_beliefQAs,
                                       assign_question_ids,generate_KJability,
                                       generate_comprehensionQA_TH,
                                       generate_justificationQA_TH,generate_fact_QA_TH,generate_1stbeliefQAs_TH,
                                       generate_2ndbeliefQAs_TH,generate_infoAccessibilityQAs_TH,
                                       generate_answerabilityQAs_TH,generate_liedetectabilityQAs_TH,
                                       generate_lieabilityQAs_TH,generate_KJability_TH)
from justification_option_generator import (generate_justificationQA_instant,generate_justificationQA_instant_TH)

from conv_generation_utils import replace_ABCD_with_name


In [ ]:
# import json

# with open(
#     "/code/generated_conversations0-th-thainame-testpersonality.json",
#     "r",
#     encoding="utf-8"
# ) as file:
#     data = json.load(file)

# print(f"Loaded {len(data)} data items")
# print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

# for i, d in enumerate(data):
#     lie = d.get("lie", {})

#     A_name = d.get("characters", {}).get("lair")
#     B_name = d.get("characters", {}).get("target")
#     C_name = d.get("characters", {}).get("accomplice")
#     D_name = d.get("characters", {}).get("observer")

#     # Replace text
#     lie["real_reason_c"] = replace_ABCD_with_name(
#         lie.get("real_reason_c", ""),
#         A_name=A_name,
#         B_name=B_name,
#         C_name=C_name,
#         D_name=D_name
#     )

#     lie["lie_c"] = replace_ABCD_with_name(
#         lie.get("lie_c", ""),
#         A_name=A_name,
#         B_name=B_name,
#         C_name=C_name,
#         D_name=D_name
#     )

#     lie["truth_c"] = replace_ABCD_with_name(
#         lie.get("truth_c", ""),
#         A_name=A_name,
#         B_name=B_name,
#         C_name=C_name,
#         D_name=D_name
#     )

#     # Assign back (optional but explicit)
#     d["lie"] = lie

# # Save updated JSON
# with open(
#     "/code/generated_conversations0-th-thainame-testpersonality.json",
#     "w",
#     encoding="utf-8"
# ) as file:
#     json.dump(data, file, ensure_ascii=False, indent=2)

# print("✅ JSON file updated successfully")


In [ ]:
# #replace name
# with open("/code/generated_conversations3-th.json", "r", encoding="utf-8") as file:
#     data = json.load(file)

# print(f"Loaded {len(data)} data items")
# print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

# for i,d in enumerate(data):

#     lie = d.get("lie", "unknown")
#     A_name=d.get("characters", {}).get("lair")
#     B_name=d.get("characters", {}).get("target")
#     C_name=d.get("characters", {}).get("accomplice")
#     D_name=d.get("characters", {}).get("observer")

#     new_real_reasons_c=replace_ABCD_with_name(lie["real_reason_c"],A_name=A_name,B_name=B_name,C_name=C_name,D_name=D_name)
#     new_lie_c=replace_ABCD_with_name(lie["lie_c"],A_name=A_name,B_name=B_name,C_name=C_name,D_name=D_name)
#     new_truth_c=replace_ABCD_with_name(lie["truth_c"],A_name=A_name,B_name=B_name,C_name=C_name,D_name=D_name)
#     print(new_lie_c,new_truth_c,new_real_reasons_c)

In [ ]:
# #add q_id to KJability
# import json

# input_path = "/dataset/generated_questions/generated_conversations0-th-testpersonality.json"
# output_path = "/dataset/generated_questions/generated_conversations0-th-testpersonality.json"

# # Load JSON
# with open(input_path, "r", encoding="utf-8") as file:
#     data = json.load(file)

# print(f"Loaded {len(data)} data items")
# print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

# # Traverse and modify
# for item_idx, d in enumerate(data):
#     set_id = d.get("set_id", "unknown")
#     kj_ability = d.get("KJability", [])

#     print(set_id)

#     # KJability should be a list of question dicts
#     for q_idx, q_dict in enumerate(kj_ability):
#         q_dict["q_id"] = f"{set_id}-KJability-{q_idx}"

#         print(q_dict.get("question", ""))
#         print(q_dict["q_id"])

# # Save updated JSON
# with open(output_path, "w", encoding="utf-8") as file:
#     json.dump(data, file, ensure_ascii=False, indent=2)

# print(f"Saved updated JSON to: {output_path}")


In [ ]:
def generate_all_questions(selected_set):
    """
    Generate all types of questions for a given dataset item.
    
    Args:
        selected_set (dict): A single item from the dataset
        
    Returns:
        dict: Combined dictionary with all question types
    """
    selected_set_id = selected_set.get("set_id", "unknown")
    print(f"Processing set: {selected_set_id}")

    # Generate all question types
    comprehension_qas = generate_comprehensionQA(selected_set)
    justification_qas = generate_justificationQA(selected_set)
    fact_qas = generate_fact_QA(selected_set)
    
    # Generate and merge belief questions
    belief_qas_1 = generate_1stbeliefQAs(selected_set)
    belief_qas_2 = generate_2ndbeliefQAs(selected_set)
    belief_qas = merge_beliefQAs(belief_qas_1, belief_qas_2)

    # Generate ToM-related questions
    info_accessibility_qas = generate_infoAccessibilityQAs(selected_set)
    answerability_qas = generate_answerabilityQAs(selected_set)
    liedetectability_qas = generate_liedetectabilityQAs(selected_set)
    lieability_qas = generate_lieabilityQAs(selected_set)

    # Combine all questions into a single dictionary
    combined_entry = {
        "set_id": selected_set_id,
        **comprehension_qas,
        **justification_qas,
        **fact_qas,
        **belief_qas,
        **info_accessibility_qas,
        **answerability_qas,
        **liedetectability_qas,
        **lieability_qas
    }

    # Assign unique question IDs
    combined_entry = assign_question_ids(combined_entry, selected_set_id)
    
    return combined_entry

In [ ]:

# with open("/code/generated_conversations3-en.json", "r", encoding="utf-8") as file:
#     data = json.load(file)

# print(f"Loaded {len(data)} data items")
# print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

# #en
# for i,d in enumerate(data):
#     selected_set_id = data[0].get("set_id", "unknown")
#     print(f"Processing set: {selected_set_id}")

#     # Generate all question types
#     comprehension_qas = generate_comprehensionQA(data[0])
#     justification_qas = generate_justificationQA_instant(data[0])
#     fact_qas = generate_fact_QA(data[0])

#     # Generate and merge belief questions
#     belief_qas_1 = generate_1stbeliefQAs(data[0])
#     belief_qas_2 = generate_2ndbeliefQAs(data[0])
#     belief_qas = merge_beliefQAs(belief_qas_1, belief_qas_2)

#     # Generate ToM-related questions
#     info_accessibility_qas = generate_infoAccessibilityQAs(data[0])
#     answerability_qas = generate_answerabilityQAs(data[0])
#     liedetectability_qas = generate_liedetectabilityQAs(data[0])
#     lieability_qas = generate_lieabilityQAs(data[0])
#     KJability_qas = generate_KJability(data[0])


#     combined_entry = {
#             **data[0],
#             **comprehension_qas,
#             **justification_qas,
#             **fact_qas,
#             **belief_qas,
#             **info_accessibility_qas,
#             **answerability_qas,
#             **liedetectability_qas,
#             **lieability_qas,
#             **KJability_qas
#         }

#     # Assign unique question IDs
#     combined_entry = assign_question_ids(combined_entry, selected_set_id)

#     with open("/dataset/generated_questions/generated_question_conv3-en.json", "w", encoding="utf-8") as outfile:
#         json.dump(combined_entry, outfile, indent=4, ensure_ascii=False)


In [ ]:
#en
import json

# Load data
with open("/code/generated_conversations0-en-translated.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(f"Loaded {len(data)} data items")
print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

all_results = []  # <-- collect results here

for i, d in enumerate(data):
    selected_set_id = d.get("set_id", f"unknown_{i}")
    print(f"Processing set: {selected_set_id}")

    # Generate all question types
    comprehension_qas = generate_comprehensionQA(d)
    justification_qas = generate_justificationQA_instant(d)
    fact_qas = generate_fact_QA(d)

    belief_qas_1 = generate_1stbeliefQAs(d)
    belief_qas_2 = generate_2ndbeliefQAs(d)
    belief_qas = merge_beliefQAs(belief_qas_1, belief_qas_2)

    info_accessibility_qas = generate_infoAccessibilityQAs(d)
    answerability_qas = generate_answerabilityQAs(d)
    liedetectability_qas = generate_liedetectabilityQAs(d)
    lieability_qas = generate_lieabilityQAs(d)
    KJability_qas = generate_KJability(d)

    combined_entry = {
        **d,
        **comprehension_qas,
        **justification_qas,
        **fact_qas,
        **belief_qas,
        **info_accessibility_qas,
        **answerability_qas,
        **liedetectability_qas,
        **lieability_qas,
        **KJability_qas
    }

    combined_entry = assign_question_ids(combined_entry, selected_set_id)

    all_results.append(combined_entry)

# Write once, orderly
output_path = "/dataset/generated_questions/generated_conversations0-en-translated.json"
with open(output_path, "w", encoding="utf-8") as outfile:
    json.dump(all_results, outfile, indent=4, ensure_ascii=False)

print(f"Saved {len(all_results)} entries to {output_path}")


In [ ]:
#th

import json

# Load data
with open("/code/generated_conversations3-th.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(f"Loaded {len(data)} data items")
print(f"Sample item keys: {list(data[0].keys()) if data else 'No data'}")

all_results = []  # <-- collect results here

for i, d in enumerate(data):
    selected_set_id = d.get("set_id", f"unknown_{i}")
    print(f"Processing set: {selected_set_id}")

    # Generate all question types
    comprehension_qas = generate_comprehensionQA_TH(d)
    justification_qas = generate_justificationQA_instant_TH(d)
    fact_qas = generate_fact_QA_TH(d)

    belief_qas_1 = generate_1stbeliefQAs_TH(d)
    belief_qas_2 = generate_2ndbeliefQAs_TH(d)
    belief_qas = merge_beliefQAs(belief_qas_1, belief_qas_2)

    info_accessibility_qas = generate_infoAccessibilityQAs_TH(d)
    answerability_qas = generate_answerabilityQAs_TH(d)
    liedetectability_qas = generate_liedetectabilityQAs_TH(d)
    lieability_qas = generate_lieabilityQAs_TH(d)
    KJability_qas = generate_KJability_TH(d)

    combined_entry = {
        **d,
        **comprehension_qas,
        **justification_qas,
        **fact_qas,
        **belief_qas,
        **info_accessibility_qas,
        **answerability_qas,
        **liedetectability_qas,
        **lieability_qas,
        **KJability_qas
    }

    combined_entry = assign_question_ids(combined_entry, selected_set_id)

    all_results.append(combined_entry)

# Write once, orderly
output_path = "/dataset/generated_questions/generated_question_conv3-th.json"
with open(output_path, "w", encoding="utf-8") as outfile:
    json.dump(all_results, outfile, indent=4, ensure_ascii=False)

print(f"Saved {len(all_results)} entries to {output_path}")


In [ ]:
# #th

# selected_set_id = data[0].get("set_id", "unknown")
# print(f"Processing set: {selected_set_id}")

# # Generate all question types
# comprehension_qas = generate_comprehensionQA_TH(data[0])
# justification_qas = generate_justificationQA_instant_TH(data[0])
# fact_qas = generate_fact_QA_TH(data[0])

# # Generate and merge belief questions
# belief_qas_1 = generate_1stbeliefQAs_TH(data[0])
# belief_qas_2 = generate_2ndbeliefQAs(data[0])
# belief_qas = merge_beliefQAs(belief_qas_1, belief_qas_2)

# # Generate ToM-related questions
# info_accessibility_qas = generate_infoAccessibilityQAs_TH(data[0])
# answerability_qas = generate_answerabilityQAs_TH(data[0])
# liedetectability_qas = generate_liedetectabilityQAs_TH(data[0])
# lieability_qas = generate_lieabilityQAs_TH(data[0])
# KJability_qas = generate_KJability_TH(data[0])

# combined_entry = {
#         **data[0],
#         **comprehension_qas,
#         **justification_qas,
#         **fact_qas,
#         **belief_qas,
#         **info_accessibility_qas,
#         **answerability_qas,
#         **liedetectability_qas,
#         **lieability_qas,
#         **KJability_qas
#     }

# # Assign unique question IDs
# combined_entry = assign_question_ids(combined_entry, selected_set_id)


# with open("sample_generated_question_conv3-en.json", "w", encoding="utf-8") as outfile:
#     json.dump(combined_entry, outfile, indent=4, ensure_ascii=False)

In [ ]:
with open("sample_generated_question_conv3-en.json", "w", encoding="utf-8") as outfile:
    json.dump(combined_entry, outfile, indent=4, ensure_ascii=False)